# Phase 3: Advanced Agricultural Machine Learning Analyses

This notebook implements three critical ML-based insights for the AgriSense platform:
1. **Temporal Trend Analysis (Forecasting)**: Predicting Nitrogen (N) levels 48 hours in advance.
2. **Anomaly Detection (System Health)**: Identifying sensor errors or "dirty" data.
3. **Correlation Analysis (Feature Importance)**: Visualizing how environmental factors influence soil context.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, TimeSeriesSplit
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score, classification_report
import warnings
warnings.filterwarnings('ignore')

# Set aesthetic parameters for plots
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries loaded successfully.")

## 1. Temporal Trend Analysis (Forecasting)
**Goal**: Predict Nitrogen (N) levels 48 hours into the future using historical patterns.

In [ ]:
# Load synthetic time-series data
df_time = pd.read_csv('thingspeak_api/synthetic_agri_data_7days.csv')
df_time['Timestamp'] = pd.to_datetime(df_time['Timestamp'])
df_time = df_time.sort_values('Timestamp')

# Feature Engineering for Time-Series
# We'll create lag features to capture depletion rates
df_time['N_lag_1'] = df_time['N'].shift(1)
df_time['N_lag_2'] = df_time['N'].shift(2)
df_time['hour'] = df_time['Timestamp'].dt.hour
df_time = df_time.dropna()

X = df_time[['N_lag_1', 'N_lag_2', 'hour']]
y = df_time['N']

# Models to compare
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=100, random_state=42)
}

print("5-Fold Cross Validation (TimeSeriesSplit):")
tscv = TimeSeriesSplit(n_splits=5)

for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=tscv, scoring='neg_mean_squared_error')
    rmse_scores = np.sqrt(-scores)
    print(f"{name} RMSE: {rmse_scores.mean():.2f} (+/- {rmse_scores.std():.2f})")

# Train best model (Random Forest often performs better for non-linear trends)
best_forecast_model = RandomForestRegressor(random_state=42)
best_forecast_model.fit(X, y)

# Simulate a 48-hour prediction loop
future_N = []
last_data = X.iloc[-1:].copy()
current_N = y.iloc[-1]

for i in range(48):
    pred = best_forecast_model.predict(last_data)[0]
    future_N.append(pred)
    # Update lags for next prediction
    last_data['N_lag_2'] = last_data['N_lag_1']
    last_data['N_lag_1'] = pred
    last_data['hour'] = (last_data['hour'] + 1) % 24

plt.plot(df_time['Timestamp'][-50:], df_time['N'][-50:], label='Historical N')
future_dates = pd.date_range(start=df_time['Timestamp'].max(), periods=49, freq='H')[1:]
plt.plot(future_dates, future_N, label='48h Forecast', linestyle='--', color='red')
plt.title("Nitrogen (N) Level: 48-Hour Forecasting Trend")
plt.xlabel("Timestamp")
plt.ylabel("Nitrogen Level")
plt.legend()
plt.show()

## 2. Anomaly Detection (System Health)
**Goal**: Identify "dirty" data or sensor malfunctions (e.g., impossible pH jumps or outliers).

In [ ]:
# Load real world dataset
df_anomaly = pd.read_csv('AgriSense_ML_Ready.csv')
features = ['N', 'P', 'K', 'temperature', 'humidity', 'ph']
data = df_anomaly[features]

# Inject synthetic anomalies to test detection (e.g., pH = 14, Temp = 100)
anomaly_indices = [10, 50, 100]
df_anomaly.loc[anomaly_indices[0], 'ph'] = 14.0
df_anomaly.loc[anomaly_indices[1], 'temperature'] = 85.0
df_anomaly.loc[anomaly_indices[2], 'humidity'] = 0.5

scaler = StandardScaler()
data_scaled = scaler.fit_transform(df_anomaly[features])

# Compare Isolation Forest vs LOF
iso_forest = IsolationForest(contamination=0.02, random_state=42)
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.02)

df_anomaly['iso_anomaly'] = iso_forest.fit_predict(data_scaled)
df_anomaly['lof_anomaly'] = lof.fit_predict(data_scaled)

# -1 indicates an anomaly
print(f"Isolation Forest detected {len(df_anomaly[df_anomaly['iso_anomaly'] == -1])} anomalies.")
print(f"Local Outlier Factor detected {len(df_anomaly[df_anomaly['lof_anomaly'] == -1])} anomalies.")

# Visualization of Anomalies (pH vs Temperature)
sns.scatterplot(data=df_anomaly, x='temperature', y='ph', hue='iso_anomaly', palette={1: 'blue', -1: 'red'})
plt.title("Sensor Health: Anomaly Detection (Isolation Forest)")
plt.show()

## 3. Correlation Analysis (Feature Importance)
**Goal**: Show how much factors like Temperature/Humidity influence Crop Suitability and NPK availability.

In [ ]:
df_corr = pd.read_csv('AgriSense_ML_Ready.csv')

# Encode the label if it's categorical
le = LabelEncoder()
df_corr['label_encoded'] = le.fit_transform(df_corr['label'])

X_corr = df_corr[['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']]
y_corr = df_corr['label_encoded']

# Models to compare for classification
models_clf = {
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Decision Tree": RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42) # Simplified for example
}

print("5-Fold Cross Validation (Accuracy):")
for name, model in models_clf.items():
    cv_scores = cross_val_score(model, X_corr, y_corr, cv=5)
    print(f"{name} Accuracy: {cv_scores.mean():.2f} (+/- {cv_scores.std():.2f})")

# Best Model Feature Importance
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_corr, y_corr)

importances = rf.feature_importances_
feature_names = X_corr.columns
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances}).sort_values('Importance', ascending=False)

sns.barplot(data=feature_importance_df, x='Importance', y='Feature', palette='viridis')
plt.title("Jaffna Context: Environmental Influence on Crop Suitability")
plt.show()

print("Insights for Farmers:")
top_f = feature_importance_df.iloc[0]['Feature']
print(f"The most influential factor for your soil context is: {top_f}")